In [ ]:
import numpy as np

from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('../data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

print(f"Loaded df of size {df.shape}")

In [ ]:
from scripts.utils import extract_unique_npcis
from scripts.data_filter import filter_dataframe

selected_campaigns = list(range(1, 20))

filtered_df = df.copy()

filtered_df = filter_dataframe(
    df=filtered_df,
    operators=[10],
    include_columns=["pci", "beam_index", "nr_arfcn", "operator_id", "rsrp"],
    campaigns=selected_campaigns,
)

print(f"Filtered df of size {filtered_df.shape}")

unique_npcis = extract_unique_npcis(filtered_df['measurements_matrix'])

print(len(unique_npcis))

In [ ]:
from scripts.utils import RF_PARAM_5G
import pandas as pd
from scripts.beamforming import get_best_beam

# Assuming filtered_df and RF_PARAM_5G are defined elsewhere
data = []
for _, row in filtered_df.iterrows():
    pci, beam = get_best_beam(row['measurements_matrix'], RF_PARAM_5G.RSRP)
    if pci is not None and beam is not None:
        data.append([row['lat'], row['lng'], pci, beam])

# Convert data into a DataFrame
data_df = pd.DataFrame(data, columns=['lat', 'lng', 'pci', 'beam_index'])


In [ ]:
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


# Use a discrete colormap with enough distinct colors
def plot_beams(df: pd.DataFrame, title: str) -> None:
    plt.figure(figsize=(12, 8))
    unique_beams = np.unique(df['beam_index'])

    cmap = plt.get_cmap("tab20", 10)  # 'tab20' provides 20 distinct colors

    # Map each unique value to a color
    color_map = {val: cmap(i) for i, val in enumerate(unique_beams)}

    for beam, group in df.groupby('beam_index'):
        plt.scatter(
            group["lat"],
            group["lng"],
            color=color_map[beam],
            label=f"Beam {beam}",
            alpha=0.8,
        )

    plt.xlabel("Latitude")
    plt.ylabel("Longitude")
    plt.title(title)
    plt.legend(title="Beam", loc="lower left")
    plt.show()


def geo_plot_points(df: pd.DataFrame, df_sec: pd.DataFrame, title: str) -> None:
    """
    Plots given locations to a map (OpenStreetMap) that is viewable in broswer.
    Generates a file called 'map.html' in the current working directory.
    :param df:
    """
    # Create a map centered around the mean location
    m = folium.Map(location=[df["lat"].mean(), df["lng"].mean()], zoom_start=20)

    unique_beams = np.unique(df['beam_index'])

    cmap = plt.get_cmap("tab20", 10)  # 'tab20' provides 20 distinct colors
    color_map = {val: mcolors.to_hex(cmap(i)) for i, val in enumerate(unique_beams)}
    legend_entries = []

    for beam, group in df.groupby('beam_index'):
        # Add CircleMarkers to the map
        color = color_map[beam]
        legend_entries.append((color, '□', f"PCI -108, Beam {beam}"))

        for _, row in group.iterrows():
            folium.RegularPolygonMarker(
                location=[row["lat"], row["lng"]],
                radius=5,  # Size of the marker
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.6,
            ).add_to(m)

    for beam, group in df_sec.groupby('beam_index'):
        # Add CircleMarkers to the map
        color = color_map[beam]
        legend_entries.append((color, 'O', f"PCI -108, Beam {beam}"))
        for _, row in group.iterrows():
            folium.CircleMarker(
                location=[row["lat"], row["lng"]],
                radius=5,  # Size of the marker
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.6,
            ).add_to(m)

    # Save the map as an HTML file and open it in the browser
    m.save(f"{title}.html")


dfs = []

df_108 = data_df[data_df['pci'] == -108]
df_109 = data_df[data_df['pci'] == -109]

geo_plot_points(df_108, df_109, title="beams")

In [ ]:
filtered_df.iloc[4]['measurements_matrix']['beam_index'].value_counts()